# 03. Model A Training — Behavior & Repayment Score

This notebook trains the XGBoost classifier that predicts repayment probability based strictly on behavioral signals.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import ROCCurveDisplay, classification_report
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import pickle
import os

# Load data
df = pd.read_csv('../backend/data/synthetic_loan_data.csv')

# Features for Model A
features = [
    'utility_payment_consistency', 
    'electricity_payment_regularity', 
    'prior_repayment_record', 
    'mobile_recharge_frequency'
]

X = df[features].copy()
y = df['loan_repaid']

# Impute prior_repayment_record nulls with -1
X['prior_repayment_record'] = X['prior_repayment_record'].fillna(-1)
# Simple imputation for others
X['utility_payment_consistency'] = X['utility_payment_consistency'].fillna(0.6)
X['electricity_payment_regularity'] = X['electricity_payment_regularity'].fillna(0.6)
X['mobile_recharge_frequency'] = X['mobile_recharge_frequency'].fillna(2.5)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Original class balance: {np.bincount(y_train)}")

# SMOTE to balance classes
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)
print(f"Balanced class balance: {np.bincount(y_res)}")

# Train XGBoost
model = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
model.fit(X_res, y_res)

# Evaluate
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
ROCCurveDisplay.from_estimator(model, X_test, y_test)
plt.title('Model A — Behavioral Repayment ROC')
plt.show()

# Save
with open('../models/model_a_behavior.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Model saved to models/model_a_behavior.pkl")